# Re-scraping GPX manquants (fichiers avec '/' dans le nom)
Cible uniquement les 127 fichiers manquants et les sauvegarde avec '-' au lieu de '/'

In [1]:
import os
import re
import time
import pandas as pd
import requests
from tqdm import tqdm
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

WORK_DIR    = '/Users/arthurdeletang/Desktop/Stage M1/Code'
GPX_DIR     = os.path.join(WORK_DIR, 'data', 'gpx_files_2')
OUTPUT_BASE = os.path.join(WORK_DIR, 'data', 'matching')
RACES_CSV   = os.path.join(WORK_DIR, 'data', 'races.csv')

/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# ── 1. Identifier les 127 fichiers manquants ──────────────────────────────────
manquants = []
for year in range(2017, 2026):
    path = os.path.join(OUTPUT_BASE, 'all', f'matching_{year}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df[df['statut'] == 'match']
        m = df[df['fichier_gpx'].str.contains(' / ', na=False)]['fichier_gpx'].unique()
        manquants.extend(m)

# Dedupliquer
manquants = list(set(manquants))
print(f'Total fichiers manquants : {len(manquants)}')

# Verifier lesquels existent deja avec '-'
a_telecharger = []
for fname in manquants:
    nom_corrige = fname.replace(' / ', ' - ')
    if not os.path.exists(os.path.join(GPX_DIR, nom_corrige)):
        a_telecharger.append(fname)

print(f'A telecharger : {len(a_telecharger)}')
for f in sorted(a_telecharger):
    print(f'  {f}')

Total fichiers manquants : 127
A telecharger : 127
  2017 4 Jours de Dunkerque / Grand Prix des Hauts-de-France Stage 1.gpx
  2017 4 Jours de Dunkerque / Grand Prix des Hauts-de-France Stage 2.gpx
  2017 4 Jours de Dunkerque / Grand Prix des Hauts-de-France Stage 3.gpx
  2017 4 Jours de Dunkerque / Grand Prix des Hauts-de-France Stage 4.gpx
  2017 4 Jours de Dunkerque / Grand Prix des Hauts-de-France Stage 5.gpx
  2017 4 Jours de Dunkerque / Grand Prix des Hauts-de-France Stage 6.gpx
  2017 Dwars Door Vlaanderen / A travers la Flandre.gpx
  2017 GP de Denain - Porte du Hainaut / Valenciennes Metropole.gpx
  2017 GP de Fourmies / La Voix du Nord.gpx
  2017 Grand Prix de la Somme / Conseil Departemental 80.gpx
  2018 4 Jours de Dunkerque / Grand Prix des Hauts-de-France Stage 1.gpx
  2018 4 Jours de Dunkerque / Grand Prix des Hauts-de-France Stage 2.gpx
  2018 4 Jours de Dunkerque / Grand Prix des Hauts-de-France Stage 3.gpx
  2018 4 Jours de Dunkerque / Grand Prix des Hauts-de-France St

In [3]:
# ── 2. Charger races.csv pour retrouver les race_id ───────────────────────────
df_races = pd.read_csv(RACES_CSV)
print(f'Races : {len(df_races)} lignes')
print(df_races.head(3))

# Construire un dict nom_course -> race_id
# Le nom dans races.csv est de la forme '2023 4 Jours de Dunkerque / Grand Prix...'
race_id_map = {}
for _, row in df_races.iterrows():
    race_id_map[str(row['name'])] = str(row['id'])

# Matcher les fichiers manquants avec les race_id
# Le fichier GPX s'appelle '{name}.gpx' donc on retire le '.gpx'
matches = {}
for fname in a_telecharger:
    race_name = fname.replace('.gpx', '')
    if race_name in race_id_map:
        matches[fname] = race_id_map[race_name]
    else:
        # Chercher une correspondance partielle
        found = False
        for name, rid in race_id_map.items():
            if race_name.lower() in name.lower() or name.lower() in race_name.lower():
                matches[fname] = rid
                found = True
                break
        if not found:
            print(f'  Race ID introuvable : {fname}')

print(f'\nMatches trouves : {len(matches)}/{len(a_telecharger)}')

Races : 15475 lignes
                                                name      id  day  month  year
0  2017 Australian Road National Championships - ITT  159479    5      1  2017
1  2017 New Zealand Road National Championships -...  173198    6      1  2017
2        2017 Australian Road National Championships  159481    8      1  2017

Matches trouves : 127/127


In [4]:
# ── 3. Fonctions scraping ─────────────────────────────────────────────────────

def get_gpx(race_id, driver):
    url = f'https://www.la-flamme-rouge.eu/maps/viewtrack/gpx/{race_id}'
    cookies = {c['name']: c['value'] for c in driver.get_cookies()}
    headers = {'User-Agent': driver.execute_script('return navigator.userAgent')}
    r = requests.get(url, cookies=cookies, headers=headers, allow_redirects=True, timeout=30)
    return r.content

def create_driver():
    options = uc.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    return uc.Chrome(options=options, headless=False, version_main=148)

def login(driver, username, password):
    driver.get('https://www.la-flamme-rouge.eu')
    print('Attente Cloudflare...')
    time.sleep(5)
    if 'Just a moment' in driver.title:
        time.sleep(10)
    print('Cloudflare passe !')
    driver.get('https://www.la-flamme-rouge.eu/ucp.php?mode=login')
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.NAME, 'username')))
    for char in username:
        driver.find_element(By.NAME, 'username').send_keys(char)
        time.sleep(0.05)
    for char in password:
        driver.find_element(By.NAME, 'password').send_keys(char)
        time.sleep(0.05)
    driver.execute_script('arguments[0].click();', driver.find_element(By.NAME, 'login'))
    time.sleep(3)
    if 'incorrect' in driver.page_source.lower():
        print('Connexion echouee')
        return False
    print('Connexion reussie !')
    return True

print('Fonctions chargees.')

Fonctions chargees.


In [6]:
# ── 4. Connexion ──────────────────────────────────────────────────────────────
driver = create_driver()
login(driver, 'DeletangArthur', 'Marseille.37@')

Attente Cloudflare...
Cloudflare passe !
Connexion reussie !


True

In [7]:
# ── 5. Telechargement des GPX manquants ───────────────────────────────────────
# Sauvegarde avec '-' au lieu de '/' dans le nom

succes = []
echecs = []

for fname, race_id in tqdm(matches.items()):
    # Nom corrige : remplacer ' / ' par ' - '
    nom_corrige = fname.replace(' / ', ' - ')
    output_path = os.path.join(GPX_DIR, nom_corrige)

    if os.path.exists(output_path):
        print(f'  Deja present : {nom_corrige}')
        succes.append(fname)
        continue

    try:
        gpx_content = get_gpx(race_id, driver)
        if b'Just a moment' in gpx_content:
            print(f'  Cloudflare bloque : {fname}')
            echecs.append(fname)
            continue
        if b'<trkpt' not in gpx_content and b'<wpt' not in gpx_content:
            print(f'  Contenu invalide : {fname}')
            echecs.append(fname)
            continue
        with open(output_path, 'wb') as f:
            f.write(gpx_content)
        print(f'  OK : {nom_corrige}')
        succes.append(fname)
    except Exception as e:
        print(f'  Erreur {fname} : {e}')
        echecs.append(fname)

    time.sleep(1)

print(f'\nSucces : {len(succes)}')
print(f'Echecs : {len(echecs)}')
if echecs:
    print('Fichiers en echec :')
    for f in echecs:
        print(f'  {f}')

  0%|          | 0/127 [00:00<?, ?it/s]

  OK : 2017 GP de Fourmies - La Voix du Nord.gpx


  1%|          | 1/127 [00:02<05:06,  2.44s/it]

  OK : 2017 GP de Denain - Porte du Hainaut - Valenciennes Metropole.gpx


  2%|▏         | 2/127 [00:05<05:18,  2.55s/it]

...

  OK : 2018 Okolo jiznich Cech - Tour of South Bohemia Stage 3.gpx


 99%|█████████▉| 126/127 [05:35<00:02,  2.86s/it]

  OK : 2018 A Travers les Hauts de France - Trophe Paris-Arras Tour Stage 2.gpx


100%|██████████| 127/127 [05:38<00:00,  2.67s/it]


Succes : 127
Echecs : 0


In [ ]:
driver.quit()
print('Driver ferme.')

# Verification finale
recuperes = 0
for fname in a_telecharger:
    nom_corrige = fname.replace(' / ', ' - ')
    if os.path.exists(os.path.join(GPX_DIR, nom_corrige)):
        recuperes += 1

print(f'Fichiers recuperes : {recuperes}/{len(a_telecharger)}')

In [ ]:
# ── 6. Mise a jour du matching ────────────────────────────────────────────────
# Mettre a jour les fichiers matching pour pointer vers les nouveaux noms

for year in range(2017, 2026):
    path = os.path.join(OUTPUT_BASE, 'all', f'matching_{year}.csv')
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    mask = df['fichier_gpx'].str.contains(' / ', na=False)
    if mask.sum() == 0:
        continue
    df.loc[mask, 'fichier_gpx'] = df.loc[mask, 'fichier_gpx'].str.replace(' / ', ' - ')
    df.to_csv(path, index=False)
    print(f'{year} : {mask.sum()} lignes mises a jour')

print('Matching mis a jour.')

In [1]:
import pandas as pd
import os

MATCH_DIR = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/matching/all'

for year in range(2017, 2026):
    path = os.path.join(MATCH_DIR, f'matching_{year}.csv')
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    mask = df['fichier_gpx'].str.contains(' / ', na=False)
    if mask.sum() == 0:
        continue
    df.loc[mask, 'fichier_gpx'] = df.loc[mask, 'fichier_gpx'].str.replace(' / ', ' - ')
    df.to_csv(path, index=False)
    print(f'{year} : {mask.sum()} lignes mises a jour')

print('Done — relance maintenant CreateDatasetFinal_v2 pour reconstruire race_features.csv')

/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


2017 : 10 lignes mises a jour
2018 : 17 lignes mises a jour
2019 : 16 lignes mises a jour
2020 : 1 lignes mises a jour
2021 : 15 lignes mises a jour
2022 : 19 lignes mises a jour
2023 : 14 lignes mises a jour
2024 : 17 lignes mises a jour
2025 : 18 lignes mises a jour
Done — relance maintenant CreateDatasetFinal_v2 pour reconstruire race_features.csv
